# Exercice 1 – Manipulation de données (churn)

In [ ]:
#Importation des bibliothéques

import sys
import numpy as np          # calcul numérique (tableaux, nombres aléatoires...)
import pandas as pd         # manipulation de données tabulaires (DataFrame)
import matplotlib.pyplot as plt  # visualisation de base
import seaborn as sns       # visualisation statistique (basée sur matplotlib)

In [ ]:
#Fixer la graine du génerateur de nombre pseoudo-aléatoires

np.random.seed(42) # 42 arbitraire, on pourrait mettre 0 ou autre chose
n = 500  # nombre de clients à générer

# Générateur de données fictives pour un problème de churn
df = pd.DataFrame({
    "client_id" : range(1, n+1),  # identifiant unique du client (1 à n)
    "age" : np.random.randint(18, 75, n),  # âge tiré uniformément entre 18 et 74 ans
    "anciennete_mois" : np.random.randint(1, 120, n),  # ancienneté du client en mois (1 à 119)
    "revenu_mensuel" : np.random.lognormal(7.8, 0.5, n).round(0),  # revenu mensuel (loi log-normale, valeurs positives et asymétriques)
    "nb_produits" : np.random.choice([1,2,3,4,5], n,
p=[0.3,0.3,0.2,0.15,0.05]),  # nombre de produits souscrits, avec probabilités décroissantes
    "segment" :
np.random.choice(["Basic","Standard","Premium"], n,
p=[0.5,0.35,0.15]),  # segment du client, tiré selon des probabilités fixées
    "churn" : np.random.choice([0, 1], n, p=[0.85, 0.15])  # variable cible : 1 = a résilié (churn), 0 = est resté (15% de churn)
})

In [ ]:
#Aperçu rapide du jeu de données : dimensions et premières lignes
print(f"Dimensions : {df.shape}")  # (nombre de lignes, nombre de colonnes)
df.head()  # affiche les 5 premières lignes

In [ ]:
#Début de la phase d'exploration des données
print(df.dtypes)  # type de chaque colonne (int, float, str...)
missing = df.isna().sum()  # nombre de valeurs manquantes par colonne
print(missing[missing > 0])  # n'affiche que les colonnes qui ont des valeurs manquantes
print(df.describe().round(1))  # statistiques descriptives (moyenne, écart-type, min, max, quartiles) des colonnes numériques

In [ ]:
#Filtrage et selection des colonnes
features = df[["age", "anciennete_mois", "revenu_mensuel"]]  # variables explicatives (X)
cible = df["churn"]  # variable à prédire (y)

clients_premium = df[df["segment"] == "Premium"]  # sous-ensemble : clients du segment Premium
churners_jeunes = df[(df["churn"] == 1) & (df["age"] < 30)]  # sous-ensemble : clients ayant résilié et âgés de moins de 30 ans
riches_fideles = df[(df["revenu_mensuel"] > 4000) &
(df["anciennete_mois"] > 36)]  # sous-ensemble : revenu > 4000 et ancienneté > 36 mois

print("\nLigne 0 via .loc :", df.loc[0, "segment"])  # accès par label (index=0, colonne "segment")
print("Ligne 0 via .iloc :", df.iloc[0, -1])  # accès par position (1ère ligne, dernière colonne)

# Exercice 2 – k-NN – Généralisation

Jeu de données : **Breast Cancer** (scikit-learn). Objectif : construire des modèles k-NN et étudier leur capacité de généralisation.

### 1. Chargement du jeu de données

In [ ]:
# Chargement du jeu de données Breast Cancer fourni par scikit-learn
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()

x = data.data    # variables explicatives (tableau numpy de 569 lignes x 30 colonnes)
y = data.target  # variable cible : 0 = malin (malignant), 1 = bénin (benign)

### 2. Inspection des données

In [ ]:
# Mise en forme dans un DataFrame pandas pour faciliter l'exploration
# (df_bc pour ne pas écraser le df de l'exercice 1)
df_bc = pd.DataFrame(x, columns=data.feature_names)
df_bc["target"] = y

print(f"Dimensions : {df_bc.shape}")  # (nombre de lignes, nombre de colonnes)
print("Classes :", data.target_names)  # noms des classes de la cible
df_bc.head()  # affiche les 5 premières lignes

In [ ]:
# Répartition des classes de la cible (effectifs par classe)
print(df_bc["target"].value_counts().rename(index={0: "Malignant", 1: "Benign"}).sort_index())

In [ ]:
# Statistiques descriptives des 10 premières variables (moyenne, écart-type, min, max)
# Le dégradé de couleurs met en évidence les écarts d'échelle entre variables
df_bc.drop(columns="target").describe().loc[["mean", "std", "min", "max"]].T.head(10).style.format("{:.2f}").background_gradient(cmap="coolwarm", axis=0)

**Observation :** les variables ont des échelles très différentes (ex. `mean smoothness` ≈ 0.1 contre `mean area` ≈ 650). Comme le k-NN repose sur des distances, il faudra normaliser les données (question 4).

### 3. Découpage en jeux d'entraînement et de test

In [ ]:
# Découpage train / test avec train_test_split (random_state=42)
from sklearn.model_selection import train_test_split

# 80 % des données pour l'entraînement, 20 % pour le test
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, shuffle=True, stratify=y
)

print("Train :", X_train.shape)  # (nombre de lignes, nombre de colonnes)
print("Test  :", X_test.shape)

**À quoi servent `random_state`, `shuffle` et `stratify` ?**

- `random_state` : fixe la graine du générateur aléatoire, pour obtenir le même découpage à chaque exécution (résultats reproductibles).
- `shuffle` : mélange les données avant le découpage (activé par défaut). Utile si les données sont triées, par exemple par classe.
- `stratify` : garde la même proportion de chaque classe dans le jeu d'entraînement et dans le jeu de test.

### 4. Normalisation des données

In [ ]:
# Normalisation avec StandardScaler (fit sur le train, transform sur le train et le test)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # calcule moyenne et écart-type sur le train, puis normalise
X_test_sc = scaler.transform(X_test)        # applique la même transformation au test (pas de fit sur le test)

### 5. Création d'un classifieur k-NN (k=5)

In [ ]:
# Création du classifieur KNeighborsClassifier avec k=5
from sklearn.neighbors import KNeighborsClassifier

knn_brut = KNeighborsClassifier(n_neighbors=5)  # modèle pour les données d'origine
knn_norm = KNeighborsClassifier(n_neighbors=5)  # modèle pour les données normalisées

### 6. Entraînement des 2 modèles

In [ ]:
# Entraînement d'un modèle sur les données d'origine et d'un autre sur les données normalisées
knn_brut.fit(X_train, y_train)     # données d'origine
knn_norm.fit(X_train_sc, y_train)  # données normalisées

### 7. Comparaison des accuracies avec et sans normalisation

In [ ]:
# Accuracy d'entraînement et de test pour chaque modèle
# .score() renvoie l'accuracy (proportion de bonnes prédictions)
resultats = pd.DataFrame({
    "train": [knn_brut.score(X_train, y_train), knn_norm.score(X_train_sc, y_train)],
    "test":  [knn_brut.score(X_test, y_test),   knn_norm.score(X_test_sc, y_test)],
}, index=["sans normalisation", "avec normalisation"])

resultats.round(3)

**Commentaire :** la normalisation améliore l'accuracy, en entraînement (≈ 0.947 → 0.974) comme en test (≈ 0.912 → 0.956).

Le k-NN compare les points avec une distance. Sans normalisation, les variables à grande échelle (ex. `mean area`) dominent le calcul et les autres variables ne comptent presque pas. Après normalisation, toutes les variables ont le même poids. On garde donc les données normalisées pour la suite.

### 8. Variation de k de 1 à 50

In [ ]:
# Boucle sur k = 1 à 50 : enregistrement des accuracies d'entraînement et de test
ks = range(1, 51)
train_acc = []  # accuracy d'entraînement pour chaque k
test_acc = []   # accuracy de test pour chaque k

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)  # nouveau modèle avec k voisins
    knn.fit(X_train_sc, y_train)               # entraînement sur les données normalisées
    train_acc.append(knn.score(X_train_sc, y_train))
    test_acc.append(knn.score(X_test_sc, y_test))

### 9. Courbes d'accuracy en fonction de k

In [ ]:
# Tracé des courbes d'accuracy (train et test) en fonction de k
plt.figure(figsize=(10, 5))
plt.plot(ks, train_acc, marker="o", markersize=3, label="Train")
plt.plot(ks, test_acc, marker="o", markersize=3, label="Test")
plt.xlabel("k (nombre de voisins)")
plt.ylabel("Accuracy")
plt.title("k-NN : accuracy en fonction de k (données normalisées)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Meilleure valeur de k sur le jeu de test
best_k = ks[test_acc.index(max(test_acc))]
print(f"Meilleur k : {best_k} (accuracy test = {max(test_acc):.3f})")

**Qu'observez-vous ?**

- **k petit (k = 1)** : accuracy de train = 1.0 (chaque point est son propre voisin) mais accuracy de test plus faible (≈ 0.94). Le modèle apprend le bruit : c'est du **surapprentissage**.
- **k intermédiaire (≈ 3 à 20)** : le test atteint ses meilleures valeurs (jusqu'à ≈ 0.98) et l'écart train/test est faible : c'est la zone de **bonne généralisation**.
- **k grand (> 25)** : les deux accuracies baissent et se stabilisent (≈ 0.95). Le modèle devient trop simple : c'est du **sous-apprentissage**.

C'est le compromis **biais / variance** : un petit k donne une forte variance, un grand k un fort biais. Remarque : le jeu de test est petit (114 lignes), donc les courbes sont un peu irrégulières. Pour bien choisir k, on utiliserait plutôt une validation croisée.